In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")

os.makedirs(TABLES_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0

tfidf_metrics_path = os.path.join(TABLES_DIR, "tfidf_baselines_metrics_v2.csv")
bert_dir = os.path.join(MODELS_DIR, "bert_fulltext_trunc256")
test_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")

tfidf_df = pd.read_csv(tfidf_metrics_path)

if "split" not in tfidf_df.columns:
    raise ValueError(f"Missing 'split' column in {tfidf_metrics_path}")

tfidf_test = tfidf_df[tfidf_df["split"].astype(str).str.lower() == "test"].copy()
if len(tfidf_test) == 0:
    raise ValueError(f"No test row found in {tfidf_metrics_path}. Expected at least one row with split='test'.")

if "macro_f1" in tfidf_test.columns:
    tfidf_best_row = tfidf_test.sort_values("macro_f1", ascending=False).iloc[0]
elif "f1_fake(1)" in tfidf_test.columns:
    tfidf_best_row = tfidf_test.sort_values("f1_fake(1)", ascending=False).iloc[0]
else:
    tfidf_best_row = tfidf_test.iloc[0]

tfidf_result = {
    "model": str(tfidf_best_row.get("model", "TFIDF")),
    "split": "test",
    "accuracy": float(tfidf_best_row.get("accuracy", np.nan)),
    "macro_f1": float(tfidf_best_row.get("macro_f1", np.nan)),
    "weighted_f1": float(tfidf_best_row.get("weighted_f1", np.nan)),
    "precision_fake(1)": float(tfidf_best_row.get("precision_fake(1)", np.nan)),
    "recall_fake(1)": float(tfidf_best_row.get("recall_fake(1)", np.nan)),
    "f1_fake(1)": float(tfidf_best_row.get("f1_fake(1)", np.nan)),
    "loss": np.nan
}

test_df = pd.read_csv(test_path, usecols=["full_text", "label"])
texts = test_df["full_text"].fillna("").astype(str).tolist()
labels = test_df["label"].astype(int).to_numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(bert_dir, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(bert_dir)
model.to(device)
model.eval()

batch_size = 16
all_preds = []
all_probs = []
total_loss = 0.0
n_batches = 0

for i in tqdm(range(0, len(texts), batch_size), desc="BERT inference"):
    batch_texts = texts[i:i+batch_size]
    batch_labels = torch.tensor(labels[i:i+batch_size], dtype=torch.long, device=device)

    enc = tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt"
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc, labels=batch_labels)
        logits = out.logits
        loss_val = out.loss

    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

    total_loss += float(loss_val.item())
    n_batches += 1

    all_preds.append(preds.detach().cpu().numpy())
    all_probs.append(probs.detach().cpu().numpy())

y_pred = np.concatenate(all_preds, axis=0)
y_prob = np.concatenate(all_probs, axis=0)

acc = accuracy_score(labels, y_pred)

p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(labels, y_pred, average="macro", zero_division=0)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, y_pred, average="weighted", zero_division=0)

p_fake, r_fake, f1_fake, _ = precision_recall_fscore_support(
    labels, y_pred, average="binary", pos_label=FAKE_LABEL, zero_division=0
)

bert_result = {
    "model": "BERT_trunc256",
    "split": "test",
    "accuracy": float(acc),
    "macro_f1": float(f1_macro),
    "weighted_f1": float(f1_w),
    "precision_fake(1)": float(p_fake),
    "recall_fake(1)": float(r_fake),
    "f1_fake(1)": float(f1_fake),
    "loss": float(total_loss / max(1, n_batches))
}

out = pd.DataFrame([tfidf_result, bert_result])

out_path = os.path.join(TABLES_DIR, "main_results_comparison.csv")
out.to_csv(out_path, index=False)

print("Saved:", out_path)
print(out)